# P10 — Pembagian Dataset Development dan Test

Notebook ini memeriksa metadata, menghitung distribusi split, memeriksa file PDF yang tersedia, dan mengekspor indeks P10. Jalankan dari root repository atau sesuaikan `REPO_ROOT`.

In [45]:
!pip -q install pymupdf

In [46]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
from pathlib import Path
import pandas as pd
import pymupdf

REPO_ROOT = Path('/content/drive/MyDrive/NLPKelompok3A-main')

INDEX_PATH = REPO_ROOT / 'data' / 'materi_index.csv'
PDF_DIRS = [
    REPO_ROOT,
    REPO_ROOT / 'data' / 'raw_materials'
]

print("Repository ditemukan:", REPO_ROOT.exists())
print("File index ditemukan:", INDEX_PATH.exists())

df = pd.read_csv(
    INDEX_PATH,
    sep=';',
    encoding='utf-8-sig',
    dtype=str
).fillna('')

print('Jumlah entri metadata:', len(df))
display(df.head())

Repository ditemukan: True
File index ditemukan: True
Jumlah entri metadata: 20


,materi_id,judul,nama_file,topik_materi,kategori_literasi_terkait,penulis,pengunggah,url_sumber,status_izin,jumlah_halaman,status_ekstraksi,split,status_menurut_metadata,catatan,status_berkas_p10,halaman_awal_unit,halaman_akhir_unit,jumlah_halaman_aktual,status_rentang_p10,catatan_p10
0,M001,Literasi Bahasa Indonesia - KATA BAKU,Literasi Bahasa Indonesia - KATA BAKU.pdf,Kata baku,Memahami Informasi,Infomasukkuliah,Belum diverifikasi,https://www.scribd.com/document/671395661/LITE...,Belum diverifikasi,2,Belum diuji,development,Layak digunakan,Nama file PDF aktual belum diberikan; isi nama...,tersedia_dan_terverifikasi,1,2,2,terverifikasi_dari_pdf,Ditetapkan sebagai data development; boleh dig...
1,M002,Literasi Bahasa Indonesia - PEMAHAMAN VISUAL,Literasi Bahasa Indonesia - PEMAHAMAN VISUAL.pdf,Pemahaman visual,Menemukan Informasi,Infomasukkuliah,Belum diverifikasi,https://www.scribd.com/document/671395661/LITE...,Belum diverifikasi,4,Belum diuji,development,Layak digunakan,Nama file PDF aktual belum diberikan; isi nama...,tersedia_dan_terverifikasi,1,4,4,terverifikasi_dari_pdf,Ditetapkan sebagai data development; boleh dig...
2,M003,Literasi Bahasa Indonesia - PERNYATAAN BENAR S...,Literasi Bahasa Indonesia - PERNYATAAN BENAR S...,Pernyataan benar-salah paragraf,Menemukan Informasi,Infomasukkuliah,Belum diverifikasi,https://www.scribd.com/document/671395661/LITE...,Belum diverifikasi,2,Belum diuji,development,Layak digunakan,Nama file PDF aktual belum diberikan; isi nama...,tersedia_dan_terverifikasi,1,2,2,terverifikasi_dari_pdf,Ditetapkan sebagai data development; boleh dig...
3,M004,Literasi Bahasa Indonesia - SIMPULAN DARI PERN...,Literasi Bahasa Indonesia - SIMPULAN DARI PERN...,Simpulan dari pernyataan,Memahami Informasi,Infomasukkuliah,Belum diverifikasi,https://www.scribd.com/document/671395661/LITE...,Belum diverifikasi,3,Belum diuji,development,Layak digunakan,Nama file PDF aktual belum diberikan; isi nama...,tersedia_dan_terverifikasi,1,3,3,terverifikasi_dari_pdf,Ditetapkan sebagai data development; boleh dig...
4,M005,Pedoman EYD,Pedoman EYD.pdf,Ejaan Bahasa Indonesia,Memahami Informasi,Balai Pustaka,Belum diverifikasi,https://luk.staff.ugm.ac.id/ta/Suwardjono/EYD.pdf,Belum diverifikasi,17,Belum diuji,development,Layak digunakan,Nama file PDF aktual belum diberikan; isi nama...,belum_tersedia_di_zip,1,17,,sementara_mengikuti_metadata_belum_diverifikasi,Ditetapkan sebagai data development; boleh dig...


## 1. Tetapkan split

Split ini mempertahankan seluruh bab M013–M019 dari satu buku pada kelompok test. Pastikan keputusan ini cocok dengan isi aktual PDF sebelum dibekukan.

In [48]:
development_ids = {f'M{i:03d}' for i in range(1, 11)}
test_ids = {f'M{i:03d}' for i in range(11, 21)}

assert len(development_ids) == 10
assert len(test_ids) == 10
assert development_ids.isdisjoint(test_ids)
assert set(df['materi_id']) == development_ids | test_ids

df['split'] = df['materi_id'].apply(lambda x: 'development' if x in development_ids else 'test')
print(df['split'].value_counts())

split
development    10
test           10
Name: count, dtype: int64


## 2. Periksa ketersediaan PDF dan jumlah halaman aktual

In [49]:
available = {}
for folder in PDF_DIRS:
    if folder.exists():
        for path in folder.rglob('*.pdf'):
            available[path.name] = path

actual_pages = {}
for name, path in available.items():
    try:
        actual_pages[name] = len(pymupdf.open(path))
    except Exception as exc:
        print('Gagal membaca:', name, exc)

df['file_available'] = df['nama_file'].apply(lambda x: x in available)
df['jumlah_halaman_aktual'] = df['nama_file'].map(actual_pages).fillna('').astype(str)
df['status_berkas_p10'] = df['file_available'].map({True:'tersedia', False:'belum_tersedia'})
display(df[['materi_id','nama_file','split','jumlah_halaman','jumlah_halaman_aktual','status_berkas_p10']])

,materi_id,nama_file,split,jumlah_halaman,jumlah_halaman_aktual,status_berkas_p10
0,M001,Literasi Bahasa Indonesia - KATA BAKU.pdf,development,2,2,tersedia
1,M002,Literasi Bahasa Indonesia - PEMAHAMAN VISUAL.pdf,development,4,4,tersedia
2,M003,Literasi Bahasa Indonesia - PERNYATAAN BENAR S...,development,2,2,tersedia
3,M004,Literasi Bahasa Indonesia - SIMPULAN DARI PERN...,development,3,3,tersedia
4,M005,Pedoman EYD.pdf,development,17,17,tersedia
5,M006,Ruangguru - Kalimat Efektif.pdf,development,5,5,tersedia
6,M007,Ruangguru - Kalimat.pdf,development,7,7,tersedia
7,M008,Ruangguru - Konjungsi.pdf,development,6,6,tersedia
8,M009,"Ruangguru - Pembentukan Kata, Frasa, dan Makna...",development,8,8,tersedia
9,M010,Ruangguru - Penulisan Kata.pdf,development,6,6,tersedia


## 3. Audit jumlah dan grup sumber

In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [51]:
print('Distribusi split:')
display(df.groupby('split').agg(jumlah_dokumen=('materi_id','count'), halaman_metadata=('jumlah_halaman', lambda s: pd.to_numeric(s, errors='coerce').sum())))

print('PDF tersedia:', int(df['file_available'].sum()), 'dari', len(df))
print('PDF belum tersedia:', int((~df['file_available']).sum()))

# Bab dari buku yang sama harus tetap berada dalam satu split.
book_chapters = df[df['materi_id'].isin([f'M{i:03d}' for i in range(13,20)])]
assert book_chapters['split'].nunique() == 1
print('Split bab M013-M019:', book_chapters['split'].iloc[0])

Distribusi split:


,jumlah_dokumen,halaman_metadata
split,,
development,10,60
test,10,627


PDF tersedia: 20 dari 20
PDF belum tersedia: 0
Split bab M013-M019: test


## 4. Simpan hasil

Ekspor hanya setelah seluruh pemeriksaan dan penyesuaian manual selesai. Jangan menandai P10 final jika masih ada PDF yang belum tersedia atau belum diperiksa.

In [52]:
OUTPUT_PATH = REPO_ROOT / 'data' / 'materi_index_p10.csv'
df.to_csv(OUTPUT_PATH, sep=';', index=False, encoding='utf-8-sig')
print('Tersimpan:', OUTPUT_PATH)

Tersimpan: /content/drive/MyDrive/NLPKelompok3A-main/data/materi_index_p10.csv
